In [ ]:
!pip install fastapi uvicorn pyngrok nest_asyncio

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install fastapi uvicorn pyngrok tensorflow scikit-learn pandas -q

In [3]:
from pyngrok import ngrok
ngrok.set_auth_token("3IoJDaYu7Q5L5MaCWgwIVRrCIHc_2cxrjf9qzbuQGWNw3Fp19")

In [4]:
import tensorflow as tf
import pandas as pd
import pickle

BASE = "/content/drive/MyDrive/infosys_internship"
model = tf.keras.models.load_model(f"{BASE}/aqi_lstm_model.keras")
with open(f"{BASE}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

df = pd.read_csv(f"{BASE}/delhi_test.csv")
station_df = df[df["StationId"] == "DL001"].sort_values("Datetime").tail(24)

print("Model expects input shape:", model.input_shape)
print("Rows found for station:", len(station_df))
print("Columns available:", list(station_df.columns))

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model expects input shape: (None, 24, 42)
Rows found for station: 24
Columns available: ['Datetime', 'StationId', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'AQI', 'AQI_Bucket', 'StationName', 'City', 'State', 'Latitude', 'Longitude', 'Temperature_C', 'Humidity_pct', 'WindSpeed_kmh', 'Pressure_hPa', 'hour', 'day_of_week', 'month', 'is_weekend', 'PM2.5_roll3h', 'PM2.5_roll24h', 'PM10_roll3h', 'PM10_roll24h', 'NO2_roll3h', 'NO2_roll24h', 'CO_roll3h', 'CO_roll24h', 'SO2_roll3h', 'SO2_roll24h', 'O3_roll3h', 'O3_roll24h', 'AQI_roll3h', 'AQI_roll24h', 'Temperature_C_roll3h', 'Temperature_C_roll24h', 'Humidity_pct_roll3h', 'Humidity_pct_roll24h', 'WindSpeed_kmh_roll3h', 'WindSpeed_kmh_roll24h']


In [20]:
import os
print(os.listdir('/content/drive/MyDrive/infosys_internship'))

['stations.csv', 'station_hour.csv', 'filter_city_data.py', 'delhi_station_hour.csv', 'add_coordinates.py', 'delhi_station_hour_with_coords.csv', 'add_weather.py', 'delhi_station_hour_with_weather.csv', 'clean_data.py', 'delhi_cleaned.csv', 'feature_engineering.py', 'delhi_features.csv', 'scale_and_split.py', 'scaler.pkl', 'delhi_train.csv', 'delhi_test.csv', 'aqi_lstm_model.keras']


In [5]:
"""
FastAPI service for Colab - wraps your trained LSTM model and exposes it
via a public ngrok URL, so you can test it from any browser.

Run this AFTER: mounting Drive, and pip install fastapi uvicorn pyngrok

In a Colab cell BEFORE this one, set your ngrok token:
    from pyngrok import ngrok
    ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN_HERE")
"""

import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pyngrok import ngrok
import threading

BASE_PATH = "/content/drive/MyDrive/infosys_internship"
SEQ_LEN = 24

app = FastAPI(title="AQI Prediction Service")

model = tf.keras.models.load_model(f"{BASE_PATH}/aqi_lstm_model.keras")
with open(f"{BASE_PATH}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

df = pd.read_csv(f"{BASE_PATH}/delhi_test.csv", low_memory=False)
df["Datetime"] = pd.to_datetime(df["Datetime"])
exclude_cols = ["StationId", "StationName", "City", "State", "Datetime", "AQI_Bucket"]
feature_cols = [c for c in df.columns if c not in exclude_cols]
target_idx = feature_cols.index("AQI")
aqi_min = scaler.data_min_[target_idx]
aqi_max = scaler.data_max_[target_idx]


@app.get("/")
def root():
    return {"message": "AQI Prediction Service is running. See /docs for usage."}


@app.get("/predict/{station_id}")
def predict(station_id: str):
    try:
        station_data = df[df["StationId"] == station_id].sort_values("Datetime")
        if len(station_data) < SEQ_LEN:
            raise HTTPException(status_code=404, detail=f"Not enough data for station '{station_id}'")

        recent_window = station_data.tail(SEQ_LEN)[feature_cols].values.astype("float32")
        recent_window = recent_window.reshape(1, SEQ_LEN, len(feature_cols))

        prediction_scaled = model.predict(recent_window, verbose=0)[0][0]
        predicted_aqi = float(prediction_scaled * (aqi_max - aqi_min) + aqi_min)
        last_timestamp = station_data["Datetime"].iloc[-1]

        return {
            "station_id": station_id,
            "predicted_aqi": round(predicted_aqi, 1),
            "based_on_data_up_to": str(last_timestamp),
        }
    except HTTPException:
        raise
    except Exception as e:
        # TEMPORARY: surfaces the real error in the response for debugging.
        # Remove this except block once things are working, since exposing
        # raw errors isn't something you'd want in a real deployed API.
        import traceback
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {e}\n{traceback.format_exc()}")


# --- Start the server + public tunnel ---
# ngrok's free tier now requires a claimed domain - get yours free at
# dashboard.ngrok.com/domains and paste it below
NGROK_DOMAIN = "material-rhyme-friend.ngrok-free.dev"
public_url = ngrok.connect(8000, domain=NGROK_DOMAIN)
print(f"Public URL: {public_url.public_url}")
print(f"Try it at: {public_url.public_url}/docs")

# Run uvicorn in a background thread instead of uvicorn.run() directly -
# nest_asyncio isn't reliably compatible with Python 3.13's asyncio, and
# this sidesteps the "event loop already running" conflict entirely.
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("Server started in background thread. Keep this cell running.")

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Public URL: https://material-rhyme-friend.ngrok-free.dev
Try it at: https://material-rhyme-friend.ngrok-free.dev/docs
Server started in background thread. Keep this cell running.


In [13]:
import numpy as np

# Scale the input using the SAME scaler used in training
X_scaled_df = pd.DataFrame(
    scaler.transform(X_df),
    columns=feature_cols
)

X = X_scaled_df.values.astype("float32").reshape(1, 24, 42)
pred_scaled = model.predict(X, verbose=0)
print("Scaled prediction:", pred_scaled)

# Now unscale it back to a real AQI number
# Need AQI's column position among feature_cols
aqi_idx = feature_cols.index("AQI")
dummy = np.zeros((1, 42))
dummy[0, aqi_idx] = pred_scaled[0, 0]
real_pred = scaler.inverse_transform(dummy)[0, aqi_idx]
print("Predicted AQI:", real_pred)

NameError: name 'X_df' is not defined

In [7]:
import numpy as np

print("Any NaN in station_df?", station_df[feature_cols].isna().any().any())
print("Any Inf in station_df?", np.isinf(station_df[feature_cols].values.astype(float)).any())
print()
print(station_df[feature_cols].describe().T[["min", "max"]])

Any NaN in station_df? False
Any Inf in station_df? False

                            min       max
PM2.5                  0.038971  0.110223
PM10                   0.056741  0.192992
NO                     0.000180  0.002440
NO2                    0.000180  0.078286
NOx                    0.000400  0.042200
NH3                    0.052095  0.083971
CO                     0.006600  0.022400
SO2                    0.043419  0.057257
O3                     0.004006  0.034900
Benzene                0.000000  0.000000
Toluene                0.013413  0.013413
AQI                    0.177686  0.338843
Latitude               0.867117  0.867117
Longitude              0.470921  0.470921
Temperature_C          0.604444  0.822222
Humidity_pct           0.565217  0.847826
WindSpeed_kmh          0.021596  0.077984
Pressure_hPa           0.242105  0.434211
hour                   0.000000  1.000000
day_of_week            0.166667  0.333333
month                  0.454545  0.545455
is_weekend       

In [9]:
model = tf.keras.models.load_model('/content/drive/MyDrive/infosys_internship/aqi_lstm_model.keras')

In [10]:
import numpy as np

nan_found = False
for layer in model.layers:
    for w in layer.get_weights():
        if np.isnan(w).any():
            print(f"NaN found in layer: {layer.name}")
            nan_found = True

print("Model has NaN weights:", nan_found)

Model has NaN weights: False


In [11]:
import os
path = '/content/drive/MyDrive/infosys_internship'
for f in os.listdir(path):
    if 'keras' in f.lower():
        size = os.path.getsize(os.path.join(path, f))
        print(f, "-", size, "bytes")

aqi_lstm_model.keras - 521816 bytes


In [12]:
X = station_df[feature_cols].values.astype("float32").reshape(1, 24, 42)
pred_scaled = model.predict(X, verbose=0)
print("Scaled prediction:", pred_scaled)

aqi_idx = feature_cols.index("AQI")
dummy = np.zeros((1, 42))
dummy[0, aqi_idx] = pred_scaled[0, 0]
real_pred = scaler.inverse_transform(dummy)[0, aqi_idx]
print("Predicted AQI:", real_pred)

Scaled prediction: [[0.1965602]]
Predicted AQI: 111.1351387500763


In [14]:
import threading
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from fastapi import FastAPI, HTTPException
import uvicorn
from pyngrok import ngrok

BASE_PATH = "/content/drive/MyDrive/infosys_internship"
NGROK_DOMAIN = "material-rhyme-friend.ngrok-free.dev"  # your claimed free domain

IDENTITY_COLS = ["Datetime", "StationId", "AQI_Bucket", "StationName", "City", "State"]

app = FastAPI(title="AQI Prediction Service")

# Load once, at startup
model = tf.keras.models.load_model(f"{BASE_PATH}/aqi_lstm_model.keras")
with open(f"{BASE_PATH}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

df = pd.read_csv(f"{BASE_PATH}/delhi_test.csv")
feature_cols = [c for c in df.columns if c not in IDENTITY_COLS]
aqi_idx = feature_cols.index("AQI")


@app.get("/")
def root():
    return {"status": "AQI Prediction API is running", "stations_available": df["StationId"].nunique()}


@app.get("/predict/{station_id}")
def predict(station_id: str):
    station_df = df[df["StationId"] == station_id].sort_values("Datetime").tail(24)

    if len(station_df) < 24:
        raise HTTPException(
            status_code=404,
            detail=f"Not enough data for station '{station_id}' (found {len(station_df)} rows, need 24)."
        )

    try:
        X_df = station_df[feature_cols].interpolate(limit_direction="both").fillna(0)
        X = X_df.values.astype("float32").reshape(1, 24, 42)

        pred_scaled = model.predict(X, verbose=0)

        dummy = np.zeros((1, 42))
        dummy[0, aqi_idx] = pred_scaled[0, 0]
        predicted_aqi = float(scaler.inverse_transform(dummy)[0, aqi_idx])

        return {
            "station_id": station_id,
            "predicted_aqi": round(predicted_aqi, 1),
            "based_on_last_timestamp": str(station_df["Datetime"].iloc[-1])
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {str(e)}")


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


ngrok.connect(8000, domain=NGROK_DOMAIN)
print("Public URL: https://" + NGROK_DOMAIN)
print("Try it at: https://" + NGROK_DOMAIN + "/docs")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("Server started in background thread. Keep this cell running.")

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Public URL: https://material-rhyme-friend.ngrok-free.dev
Try it at: https://material-rhyme-friend.ngrok-free.dev/docs
Server started in background thread. Keep this cell running.


INFO:     Started server process [689]


In [15]:
station_coords = df[["StationId", "StationName", "Latitude", "Longitude"]].drop_duplicates()

def predict_aqi_for_station(station_id):
    # exact same logic as your working /predict endpoint, just returning the raw number
    ...

@app.get("/predict_at_location")
def predict_at_location(lat: float, lon: float):
    result = idw_interpolate(lat, lon, station_coords, predict_aqi_for_station, n=4)
    return result

In [16]:
import numpy as np
import pandas as pd


def haversine_distance(lat1, lon1, lat2, lon2):
    """Distance in km between two lat/long points."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


def get_nearest_stations(target_lat, target_lon, station_coords_df, n=4):
    """Returns the n nearest stations with distances, sorted nearest first."""
    df = station_coords_df.copy()
    df["distance_km"] = df.apply(
        lambda row: haversine_distance(target_lat, target_lon, row["Latitude"], row["Longitude"]),
        axis=1
    )
    return df.sort_values("distance_km").head(n).reset_index(drop=True)


def idw_interpolate(target_lat, target_lon, station_coords_df, predict_fn, n=4, power=2):
    """
    Estimates AQI at (target_lat, target_lon) using Inverse Distance Weighting
    over the n nearest stations' predicted AQI values.
    """
    nearest = get_nearest_stations(target_lat, target_lon, station_coords_df, n=n)

    if nearest.iloc[0]["distance_km"] < 0.05:  # ~50 meters - essentially at a station
        station_id = nearest.iloc[0]["StationId"]
        aqi = predict_fn(station_id)
        return {
            "interpolated_aqi": round(aqi, 1),
            "method": "exact_station_match",
            "nearest_station": nearest.iloc[0]["StationName"],
            "distance_km": round(nearest.iloc[0]["distance_km"], 3),
            "contributing_stations": [
                {"station": nearest.iloc[0]["StationName"], "distance_km": round(nearest.iloc[0]["distance_km"], 3), "weight": 1.0, "predicted_aqi": round(aqi, 1)}
            ]
        }

    weights = []
    predictions = []
    contributions = []

    for _, row in nearest.iterrows():
        dist = row["distance_km"]
        weight = 1.0 / (dist ** power)
        aqi = predict_fn(row["StationId"])

        weights.append(weight)
        predictions.append(aqi)
        contributions.append({
            "station": row["StationName"],
            "distance_km": round(dist, 2),
            "predicted_aqi": round(aqi, 1)
        })

    weights = np.array(weights)
    predictions = np.array(predictions)
    normalized_weights = weights / weights.sum()

    interpolated_aqi = float(np.sum(normalized_weights * predictions))

    for c, w in zip(contributions, normalized_weights):
        c["weight"] = round(float(w), 3)

    return {
        "interpolated_aqi": round(interpolated_aqi, 1),
        "method": "idw",
        "n_stations_used": n,
        "contributing_stations": contributions
    }


print("Spatial interpolation functions loaded.")

Spatial interpolation functions loaded.


In [17]:
IDENTITY_COLS = ["Datetime", "StationId", "AQI_Bucket", "StationName", "City", "State"]

def predict_aqi_for_station(station_id):
    """Wraps the exact prediction logic that's already confirmed working."""
    station_df = df[df["StationId"] == station_id].sort_values("Datetime").tail(24)
    if len(station_df) < 24:
        raise ValueError(f"Not enough data for station {station_id}")

    X_df = station_df[feature_cols].interpolate(limit_direction="both").fillna(0)
    X = X_df.values.astype("float32").reshape(1, 24, 42)

    pred_scaled = model.predict(X, verbose=0)

    dummy = np.zeros((1, 42))
    dummy[0, aqi_idx] = pred_scaled[0, 0]
    return float(scaler.inverse_transform(dummy)[0, aqi_idx])


# Build the station coordinates lookup (one row per station, not per reading)
station_coords = df[["StationId", "StationName", "Latitude", "Longitude"]].drop_duplicates().reset_index(drop=True)

print("Helper ready. Stations available:", len(station_coords))

Helper ready. Stations available: 38


In [18]:
# Test point: roughly central Delhi, between a couple of known stations
result = idw_interpolate(28.60, 77.21, station_coords, predict_aqi_for_station, n=4)

print("Interpolated AQI:", result["interpolated_aqi"])
print("Method:", result["method"])
print("Contributing stations:")
for c in result["contributing_stations"]:
    print(" ", c)

Interpolated AQI: 80.1
Method: idw
Contributing stations:
  {'station': 'Vivek Vihar, Delhi - DPCC', 'distance_km': 8636.94, 'predicted_aqi': 94.6, 'weight': 0.25}
  {'station': 'IHBAS, Dilshad Garden, Delhi - CPCB', 'distance_km': 8637.09, 'predicted_aqi': 80.7, 'weight': 0.25}
  {'station': 'Anand Vihar, Delhi - DPCC', 'distance_km': 8639.3, 'predicted_aqi': 125.9, 'weight': 0.25}
  {'station': 'East Arjun Nagar, Delhi - CPCB', 'distance_km': 8643.82, 'predicted_aqi': 19.2, 'weight': 0.25}


In [19]:
# Load REAL (unscaled) coordinates - not from the scaled test data
coords_df = pd.read_csv(f"{BASE_PATH}/delhi_station_hour_with_coords.csv")
station_coords = coords_df[["StationId", "StationName", "Latitude", "Longitude"]].drop_duplicates().reset_index(drop=True)

print(station_coords.head())
print("Latitude range:", station_coords["Latitude"].min(), "-", station_coords["Latitude"].max())

  StationId                StationName   Latitude  Longitude
0     DL001       Alipur, Delhi - DPCC  28.797312  77.138667
1     DL002  Anand Vihar, Delhi - DPCC  28.646616  77.315532
2     DL003  Ashok Vihar, Delhi - DPCC  28.688517  77.173934
3     DL004     Aya Nagar, Delhi - IMD  28.476491  77.132913
4     DL005       Bawana, Delhi - DPCC  28.799660  77.032885
Latitude range: 28.4764911 - 28.8464766


In [20]:
result = idw_interpolate(28.60, 77.21, station_coords, predict_aqi_for_station, n=4)

print("Interpolated AQI:", result["interpolated_aqi"])
print("Method:", result["method"])
print("Contributing stations:")
for c in result["contributing_stations"]:
    print(" ", c)

Interpolated AQI: 94.4
Method: idw
Contributing stations:
  {'station': 'Jawaharlal Nehru Stadium, Delhi - DPCC', 'distance_km': 2.95, 'predicted_aqi': 98.2, 'weight': 0.29}
  {'station': 'Lodhi Road, Delhi - IMD', 'distance_km': 2.98, 'predicted_aqi': 104.5, 'weight': 0.284}
  {'station': 'Major Dhyan Chand National Stadium, Delhi - DPCC', 'distance_km': 3.02, 'predicted_aqi': 90.1, 'weight': 0.278}
  {'station': 'Mandir Marg, Delhi - DPCC', 'distance_km': 4.13, 'predicted_aqi': 75.3, 'weight': 0.148}


In [21]:
from fastapi import HTTPException

@app.get("/predict_at_location")
def predict_at_location(lat: float, lon: float, n_stations: int = 4):
    try:
        result = idw_interpolate(lat, lon, station_coords, predict_aqi_for_station, n=n_stations)
        result["query_location"] = {"lat": lat, "lon": lon}
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {str(e)}")

print("New route added: /predict_at_location?lat=..&lon=..")

New route added: /predict_at_location?lat=..&lon=..


In [22]:
import requests

ORS_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjJlNzVhZTExMDY1NDQ0ZWE4MWI1MTg5OWM3NWNmYzk3IiwiaCI6Im11cm11cjY0In0="
  # same key already used in track_live_data.py

def sample_route_points(coords, interval_km=2.0):
    """coords: list of [lon, lat] pairs from ORS. Returns a thinned-out list, ~1 point per interval_km."""
    sampled = [coords[0]]
    accumulated = 0.0
    for i in range(1, len(coords)):
        lon1, lat1 = coords[i - 1]
        lon2, lat2 = coords[i]
        accumulated += haversine_distance(lat1, lon1, lat2, lon2)
        if accumulated >= interval_km:
            sampled.append(coords[i])
            accumulated = 0.0
    if sampled[-1] != coords[-1]:
        sampled.append(coords[-1])
    return sampled


def get_route_geometry(start_lat, start_lon, end_lat, end_lon):
    """Calls OpenRouteService, returns list of [lon, lat] points along the route + total distance/duration."""
    url = "https://api.heigit.org/openrouteservice/v2/directions/driving-car"
    params = {
        "api_key": ORS_API_KEY,
        "start": f"{start_lon},{start_lat}",
        "end": f"{end_lon},{end_lat}",
    }
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    feature = data["features"][0]
    coords = feature["geometry"]["coordinates"]  # [[lon, lat], ...]
    summary = feature["properties"]["summary"]
    return coords, summary["distance"] / 1000, summary["duration"] / 60  # km, minutes


@app.get("/route_exposure")
def route_exposure(start_lat: float, start_lon: float, end_lat: float, end_lon: float, sample_interval_km: float = 2.0):
    try:
        coords, distance_km, duration_min = get_route_geometry(start_lat, start_lon, end_lat, end_lon)
        sampled_points = sample_route_points(coords, interval_km=sample_interval_km)

        point_predictions = []
        for lon, lat in sampled_points:
            result = idw_interpolate(lat, lon, station_coords, predict_aqi_for_station, n=4)
            point_predictions.append({
                "lat": round(lat, 5),
                "lon": round(lon, 5),
                "predicted_aqi": result["interpolated_aqi"]
            })

        aqi_values = [p["predicted_aqi"] for p in point_predictions]
        avg_exposure = round(sum(aqi_values) / len(aqi_values), 1)
        max_exposure = round(max(aqi_values), 1)

        return {
            "route_distance_km": round(distance_km, 2),
            "route_duration_min": round(duration_min, 1),
            "points_sampled": len(point_predictions),
            "average_aqi_exposure": avg_exposure,
            "peak_aqi_exposure": max_exposure,
            "route_points": point_predictions
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {str(e)}")

print("New route added: /route_exposure")

New route added: /route_exposure


In [23]:
def get_aqi_category(aqi):
    if aqi <= 50:
        return "Good", "Minimal impact. Safe for outdoor activities."
    elif aqi <= 100:
        return "Satisfactory", "Minor breathing discomfort to sensitive people."
    elif aqi <= 200:
        return "Moderate", "Breathing discomfort to people with lung/heart disease, children, and elderly."
    elif aqi <= 300:
        return "Poor", "Breathing discomfort on prolonged exposure. Avoid outdoor exertion."
    elif aqi <= 400:
        return "Very Poor", "Respiratory illness on prolonged exposure. Avoid outdoor activity."
    else:
        return "Severe", "Affects healthy people; seriously impacts those with existing conditions. Avoid all outdoor exposure."


def get_confidence_interval(predicted_aqi, rmse=8.2, z=1.96):
    """Approximate ~95% interval using the model's own test-set RMSE as the error margin."""
    margin = round(z * rmse, 1)
    return {
        "lower_bound": max(0, round(predicted_aqi - margin, 1)),
        "upper_bound": round(predicted_aqi + margin, 1),
        "margin": margin,
        "confidence_level": "~95% (approximated from model's test RMSE, not a statistical guarantee)"
    }


def get_pollutant_snapshot(station_id):
    """Returns the latest known (not forecasted) pollutant readings feeding into the prediction."""
    station_df = df[df["StationId"] == station_id].sort_values("Datetime").tail(24)
    latest_row_scaled = station_df[feature_cols].iloc[[-1]].interpolate(limit_direction="both").fillna(0)
    unscaled = scaler.inverse_transform(latest_row_scaled.values)[0]

    pollutant_names = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene"]
    idxs = {name: feature_cols.index(name) for name in pollutant_names}

    return {name: round(float(unscaled[idx]), 2) for name, idx in idxs.items()}


@app.get("/predict/{station_id}")
def predict_enriched(station_id: str):
    """Enriched version: predicted AQI + health category + confidence interval + pollutant snapshot."""
    station_df = df[df["StationId"] == station_id].sort_values("Datetime").tail(24)
    if len(station_df) < 24:
        raise HTTPException(status_code=404, detail=f"Not enough data for station '{station_id}'.")

    try:
        predicted_aqi = predict_aqi_for_station(station_id)
        category, advisory = get_aqi_category(predicted_aqi)
        ci = get_confidence_interval(predicted_aqi)
        pollutants = get_pollutant_snapshot(station_id)

        return {
            "station_id": station_id,
            "predicted_aqi": round(predicted_aqi, 1),
            "health_category": category,
            "health_advisory": advisory,
            "confidence_interval": ci,
            "latest_pollutant_readings": pollutants,
            "note": "Pollutant readings are the latest known values used as model input, not a per-pollutant forecast.",
            "based_on_last_timestamp": str(station_df["Datetime"].iloc[-1])
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {str(e)}")

print("Enriched /predict/{station_id} endpoint updated.")

Enriched /predict/{station_id} endpoint updated.


In [24]:
# Remove any existing /predict/{station_id} routes before re-adding the enriched one
app.router.routes = [
    r for r in app.router.routes
    if not (hasattr(r, "path") and r.path == "/predict/{station_id}")
]

@app.get("/predict/{station_id}")
def predict_enriched(station_id: str):
    station_df = df[df["StationId"] == station_id].sort_values("Datetime").tail(24)
    if len(station_df) < 24:
        raise HTTPException(status_code=404, detail=f"Not enough data for station '{station_id}'.")

    try:
        predicted_aqi = predict_aqi_for_station(station_id)
        category, advisory = get_aqi_category(predicted_aqi)
        ci = get_confidence_interval(predicted_aqi)
        pollutants = get_pollutant_snapshot(station_id)

        return {
            "station_id": station_id,
            "predicted_aqi": round(predicted_aqi, 1),
            "health_category": category,
            "health_advisory": advisory,
            "confidence_interval": ci,
            "latest_pollutant_readings": pollutants,
            "note": "Pollutant readings are the latest known values used as model input, not a per-pollutant forecast.",
            "based_on_last_timestamp": str(station_df["Datetime"].iloc[-1])
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {str(e)}")

print("Old route removed, enriched /predict/{station_id} registered fresh.")

Old route removed, enriched /predict/{station_id} registered fresh.


In [25]:
#ADD WHETHER + TRAFFIC TO API
# --- Enrich /predict/{station_id} with weather ---
def get_weather_snapshot(station_id):
    station_df = df[df["StationId"] == station_id].sort_values("Datetime").tail(24)
    latest = station_df[feature_cols].iloc[[-1]].interpolate(limit_direction="both").fillna(0)
    unscaled = scaler.inverse_transform(latest.values)[0]
    names = ["Temperature_C", "Humidity_pct", "WindSpeed_kmh", "Pressure_hPa"]
    idxs = {n: feature_cols.index(n) for n in names}
    return {n: round(float(unscaled[i]), 2) for n, i in idxs.items()}


# Remove old /predict/{station_id} route before redefining
app.router.routes = [
    r for r in app.router.routes
    if not (hasattr(r, "path") and r.path == "/predict/{station_id}")
]

@app.get("/predict/{station_id}")
def predict_enriched(station_id: str):
    station_df = df[df["StationId"] == station_id].sort_values("Datetime").tail(24)
    if len(station_df) < 24:
        raise HTTPException(status_code=404, detail=f"Not enough data for station '{station_id}'.")
    try:
        predicted_aqi = predict_aqi_for_station(station_id)
        category, advisory = get_aqi_category(predicted_aqi)
        ci = get_confidence_interval(predicted_aqi)
        pollutants = get_pollutant_snapshot(station_id)
        weather = get_weather_snapshot(station_id)

        return {
            "station_id": station_id,
            "predicted_aqi": round(predicted_aqi, 1),
            "health_category": category,
            "health_advisory": advisory,
            "confidence_interval": ci,
            "latest_pollutant_readings": pollutants,
            "latest_weather": weather,
            "note": "Pollutant and weather readings are the latest known values used as model input, not a forecast.",
            "based_on_last_timestamp": str(station_df["Datetime"].iloc[-1])
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {str(e)}")


# --- Enrich /route_exposure with a real traffic classification ---
def classify_traffic(distance_km, duration_min):
    """
    Compares actual speed to a 'free-flow' baseline for Delhi city roads (~35 km/h)
    to estimate congestion level. This is a standard, defensible proxy approach
    given no direct live traffic-density API was used.
    """
    if duration_min <= 0:
        return "Unknown", None
    actual_speed_kmh = distance_km / (duration_min / 60)
    FREE_FLOW_SPEED = 35.0  # reasonable average for Delhi arterial roads
    ratio = actual_speed_kmh / FREE_FLOW_SPEED

    if ratio >= 0.85:
        level = "Light"
    elif ratio >= 0.6:
        level = "Moderate"
    else:
        level = "Heavy"

    return level, round(actual_speed_kmh, 1)


app.router.routes = [
    r for r in app.router.routes
    if not (hasattr(r, "path") and r.path == "/route_exposure")
]

@app.get("/route_exposure")
def route_exposure_enriched(start_lat: float, start_lon: float, end_lat: float, end_lon: float, sample_interval_km: float = 2.0):
    try:
        coords, distance_km, duration_min = get_route_geometry(start_lat, start_lon, end_lat, end_lon)
        sampled_points = sample_route_points(coords, interval_km=sample_interval_km)

        point_predictions = []
        for lon, lat in sampled_points:
            r = idw_interpolate(lat, lon, station_coords, predict_aqi_for_station, n=4)
            point_predictions.append({"lat": round(lat, 5), "lon": round(lon, 5), "predicted_aqi": r["interpolated_aqi"]})

        aqi_values = [p["predicted_aqi"] for p in point_predictions]
        traffic_level, avg_speed = classify_traffic(distance_km, duration_min)

        # nearest station's weather as a stand-in for "route weather" (single city, weather doesn't vary much)
        nearest_station = get_nearest_stations(start_lat, start_lon, station_coords, n=1).iloc[0]["StationId"]
        weather = get_weather_snapshot(nearest_station)

        return {
            "route_distance_km": round(distance_km, 2),
            "route_duration_min": round(duration_min, 1),
            "traffic_level": traffic_level,
            "average_speed_kmh": avg_speed,
            "weather": weather,
            "points_sampled": len(point_predictions),
            "average_aqi_exposure": round(sum(aqi_values) / len(aqi_values), 1),
            "peak_aqi_exposure": round(max(aqi_values), 1),
            "route_points": point_predictions
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {str(e)}")

print("Both endpoints updated: /predict now includes weather, /route_exposure now includes traffic level + weather.")

Both endpoints updated: /predict now includes weather, /route_exposure now includes traffic level + weather.


In [26]:

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="AQI Prediction Service")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def root():
    return {"status": "AQI Prediction API is running", "stations_available": df["StationId"].nunique()}

# ... then paste in all three endpoint definitions (predict_enriched, predict_at_location, route_exposure_enriched)
# exactly as you had them working before

In [27]:
"""
Weekly retraining pipeline for the AQI LSTM forecasting model.

Purpose: as new sensor data accumulates (e.g. from the live tracking
pipeline, once it has been running long enough to build proper 24-hour
sequences), this script fine-tunes the existing model on the new batch
rather than retraining from scratch - faster, and keeps what the model
already learned.

Expects:
  - A NEW_DATA_PATH CSV with the SAME columns as delhi_features.csv
    (i.e. already cleaned + feature-engineered, but NOT yet scaled -
    real units, same as delhi_features.csv).
  - The existing trained model (aqi_lstm_model.keras) and the
    ORIGINAL scaler (scaler.pkl) - the new data is scaled using the
    EXISTING scaler (transform, not fit_transform), so old and new
    data stay on the same scale.

Run this whenever a new batch of labeled data is ready (weekly, or
whenever there's enough new data to be worth a retrain).
"""

import os
import pickle
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf

BASE_PATH = "."  # adjust to your Drive/folder path when running
MODEL_PATH = os.path.join(BASE_PATH, "aqi_lstm_model.keras")
SCALER_PATH = os.path.join(BASE_PATH, "scaler.pkl")
NEW_DATA_PATH = os.path.join(BASE_PATH, "new_data_batch.csv")  # <-- point this at your new data
TEST_DATA_PATH = os.path.join(BASE_PATH, "delhi_test.csv")  # held-out set, for before/after comparison

IDENTITY_COLS = ["Datetime", "StationId", "AQI_Bucket", "StationName", "City", "State"]
SEQUENCE_LENGTH = 24
FINE_TUNE_EPOCHS = 5
FINE_TUNE_LR = 0.0001  # low learning rate - fine-tuning, not training from scratch

LOG_PATH = os.path.join(BASE_PATH, "retraining_log.csv")


def build_sequences(df, feature_cols, aqi_idx, seq_len=24):
    """Groups by station, builds (X, y) sequences: past seq_len hours -> next AQI value."""
    X_list, y_list = [], []
    for station_id, group in df.groupby("StationId"):
        group = group.sort_values("Datetime")
        values = group[feature_cols].values.astype("float32")
        if len(values) <= seq_len:
            continue
        for i in range(len(values) - seq_len):
            X_list.append(values[i:i + seq_len])
            y_list.append(values[i + seq_len, aqi_idx])
    return np.array(X_list), np.array(y_list)


def evaluate(model, X, y, scaler, feature_cols, aqi_idx):
    """Returns MAE and RMSE in real AQI units (unscaled)."""
    preds_scaled = model.predict(X, verbose=0).flatten()

    def unscale_aqi(vals):
        dummy = np.zeros((len(vals), len(feature_cols)))
        dummy[:, aqi_idx] = vals
        return scaler.inverse_transform(dummy)[:, aqi_idx]

    preds_real = unscale_aqi(preds_scaled)
    y_real = unscale_aqi(y)

    mae = float(np.mean(np.abs(preds_real - y_real)))
    rmse = float(np.sqrt(np.mean((preds_real - y_real) ** 2)))
    return mae, rmse


def main():
    print(f"=== Retraining run started: {datetime.now().isoformat()} ===")

    if not os.path.exists(NEW_DATA_PATH):
        print(f"No new data batch found at {NEW_DATA_PATH}. Nothing to retrain on. Exiting.")
        print("(This is expected until enough new live-tracked data has accumulated to build 24-hour sequences.)")
        return

    print("Loading existing model and scaler...")
    model = tf.keras.models.load_model(MODEL_PATH)
    with open(SCALER_PATH, "rb") as f:
        scaler = pickle.load(f)

    print("Loading new data batch...")
    new_df = pd.read_csv(NEW_DATA_PATH)
    feature_cols = [c for c in new_df.columns if c not in IDENTITY_COLS]
    aqi_idx = feature_cols.index("AQI")

    if len(feature_cols) != model.input_shape[-1]:
        raise ValueError(
            f"Column mismatch: new data has {len(feature_cols)} feature columns, "
            f"model expects {model.input_shape[-1]}. Check that new_data_batch.csv "
            f"has the same schema as delhi_features.csv."
        )

    # Scale new data using the EXISTING scaler - do not refit
    new_df_scaled = new_df.copy()
    new_df_scaled[feature_cols] = scaler.transform(new_df[feature_cols])

    X_new, y_new = build_sequences(new_df_scaled, feature_cols, aqi_idx, SEQUENCE_LENGTH)
    print(f"Built {len(X_new)} new training sequences from the new batch.")

    if len(X_new) == 0:
        print("Not enough new data to build even one 24-hour sequence per station. Exiting.")
        return

    # Load held-out test set for before/after comparison
    test_df = pd.read_csv(TEST_DATA_PATH)
    X_test, y_test = build_sequences(test_df, feature_cols, aqi_idx, SEQUENCE_LENGTH)

    print("Evaluating model BEFORE fine-tuning...")
    mae_before, rmse_before = evaluate(model, X_test, y_test, scaler, feature_cols, aqi_idx)
    print(f"  Before: MAE={mae_before:.2f}, RMSE={rmse_before:.2f}")

    print(f"Fine-tuning on new data for {FINE_TUNE_EPOCHS} epochs (lr={FINE_TUNE_LR})...")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR), loss="mse")
    model.fit(X_new, y_new, epochs=FINE_TUNE_EPOCHS, batch_size=64, verbose=1)

    print("Evaluating model AFTER fine-tuning...")
    mae_after, rmse_after = evaluate(model, X_test, y_test, scaler, feature_cols, aqi_idx)
    print(f"  After:  MAE={mae_after:.2f}, RMSE={rmse_after:.2f}")

    # Only save the new version if it didn't get meaningfully worse
    regressed = mae_after > mae_before * 1.10  # allow up to 10% wiggle room
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    if regressed:
        print(f"WARNING: model got noticeably worse after fine-tuning (MAE {mae_before:.2f} -> {mae_after:.2f}).")
        print("NOT overwriting the production model. Saving the fine-tuned version separately for inspection.")
        new_model_path = os.path.join(BASE_PATH, f"aqi_lstm_model_REJECTED_{timestamp}.keras")
        model.save(new_model_path)
    else:
        print("Fine-tuned model looks fine or improved - saving as the new production model.")
        versioned_path = os.path.join(BASE_PATH, f"aqi_lstm_model_{timestamp}.keras")
        model.save(versioned_path)
        # keep the "live" filename pointing at the latest good model, but preserve the old one too
        shutil.copy(MODEL_PATH, os.path.join(BASE_PATH, f"aqi_lstm_model_PREVIOUS_{timestamp}.keras"))
        model.save(MODEL_PATH)  # overwrite the live model
        print(f"Live model updated: {MODEL_PATH}")
        print(f"Previous version backed up as: aqi_lstm_model_PREVIOUS_{timestamp}.keras")

    # Log this run
    log_entry = pd.DataFrame([{
        "timestamp": datetime.now().isoformat(),
        "new_sequences_used": len(X_new),
        "mae_before": round(mae_before, 3),
        "mae_after": round(mae_after, 3),
        "rmse_before": round(rmse_before, 3),
        "rmse_after": round(rmse_after, 3),
        "accepted": not regressed
    }])
    if os.path.exists(LOG_PATH):
        log_entry.to_csv(LOG_PATH, mode="a", header=False, index=False)
    else:
        log_entry.to_csv(LOG_PATH, index=False)
    print(f"Logged this run to {LOG_PATH}")

    print(f"=== Retraining run finished: {datetime.now().isoformat()} ===")


if __name__ == "__main__":
    main()

=== Retraining run started: 2026-09-11T10:32:14.571392 ===
No new data batch found at ./new_data_batch.csv. Nothing to retrain on. Exiting.
(This is expected until enough new live-tracked data has accumulated to build 24-hour sequences.)
